In [0]:
#Camada Bronze
from pyspark.sql.functions import col

df = spark.read.csv(
    "/Volumes/workspace/default/arquivos_eng_dados/train.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)
df = df.toDF("row_id","order_id", "order_date","ship_date","ship_mode","customer_id","customer_name","segment","country","city","state","postal_code","region","product_id","category","sub_category","product_name","sales")

df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("bronze_sales")

In [0]:
%sql
SELECT * 
FROM bronze_sales
LIMIT 10


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94
3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62
4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368
6,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86
7,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28
8,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152
9,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504
10,CA-2015-115812,2015-06-09,2015-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9


In [0]:
%sql
--Criando Dimensões
--A primeira é a dimensão de cliente
CREATE TABLE dim_customer AS
SELECT DISTINCT
    customer_id,
    customer_name,
    segment,
    country,
    city,
    state,
    region
FROM bronze_sales

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Dimensão Produto
CREATE TABLE dim_product AS
SELECT DISTINCT
    product_id,
    category,
    sub_category,
    product_name
FROM bronze_sales
  

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Dimensão Tempo
CREATE TABLE dim_time AS
SELECT DISTINCT
  order_date, 
   YEAR(order_date) AS Year,
   MONTH(order_date) AS Month,
   DAY(order_date) AS Day,
   QUARTER(order_date) AS Quarter,
   WEEKOFYEAR(order_date) AS WeekOfYear,
   DAYOFWEEK(order_date) AS DayOfWeek
FROM bronze_sales
    



In [0]:
%sql
-- Criando a Fato vendas
CREATE TABLE fact_sales AS
SELECT 
   row_id,
    order_id,
    order_date,
    customer_id,
    product_id,
    CAST(sales AS DOUBLE) as sales
FROM bronze_sales

num_affected_rows,num_inserted_rows


<h2>Criando As métricas Análiticas</h2>

####Receita Mensal

In [0]:
%sql
--Receita Mensal
SELECT
    YEAR(order_date) as ano,
    MONTH(order_date) as mes,
    ROUND(SUM(sales),2) as receita
FROM fact_sales
GROUP BY  YEAR(order_date) , MONTH(order_date)
order by YEAR(order_date) , MONTH(order_date)


ano,mes,receita
2015,1,14205.71
2015,2,4519.89
2015,3,55205.8
2015,4,27906.85
2015,5,23644.3
2015,6,34322.94
2015,7,33781.54
2015,8,27117.54
2015,9,81623.53
2015,10,31453.39


####Média de Vendas

In [0]:
%sql
SELECT
    YEAR(order_date) as ano,
    MONTH(order_date) as mes,
    ROUND(AVG(sales),2) as receita
FROM fact_sales
GROUP BY  YEAR(order_date) , MONTH(order_date)
order by YEAR(order_date) , MONTH(order_date)

ano,mes,receita
2015,1,184.49
2015,2,98.26
2015,3,358.48
2015,4,214.67
2015,5,195.41
2015,6,262.01
2015,7,237.9
2015,8,185.74
2015,9,305.71
2015,10,197.82


####TOP 10 Produtos mais vendidos

In [0]:
%sql
SELECT
    p.product_name,
    ROUND(SUM(f.sales),2) as receita
FROM fact_sales f
JOIN dim_product p
ON f.product_id = p.product_id
GROUP BY p.product_name
ORDER BY receita DESC
LIMIT 10

product_name,receita
Canon imageCLASS 2200 Advanced Copier,61599.82
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27453.38
Cisco TelePresence System EX90 Videoconferencing Unit,22638.48
HON 5400 Series Task Chairs for Big and Tall,21870.58
GBC DocuBind TL300 Electric Binding System,19823.48
GBC Ibimaster 500 Manual ProClick Binding System,19024.5
Hewlett Packard LaserJet 3310 Copier,18839.69
"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",18374.9
GBC DocuBind P400 Electric Binding System,17965.07
High Speed Automatic Electric Letter Opener,17030.31


####Ranking produtos

In [0]:
%sql
SELECT
    product_id,
    ROUND(SUM(sales),2) as receita,
    RANK() OVER (ORDER BY SUM(sales) DESC) as ranking
FROM fact_sales
GROUP BY product_id

product_id,receita,ranking
TEC-CO-10004722,61599.82,1
OFF-BI-10003527,27453.38,2
TEC-MA-10002412,22638.48,3
FUR-CH-10002024,21870.58,4
OFF-BI-10001359,19823.48,5
OFF-BI-10000545,19024.5,6
TEC-CO-10001449,18839.69,7
TEC-MA-10001127,18374.9,8
OFF-BI-10004995,17965.07,9
OFF-SU-10000151,17030.31,10


####Crescimento Month over Month

In [0]:
%sql
SELECT
    ano,
    mes,
    receita,
    receita - LAG(receita) OVER (ORDER BY ano,mes) as crescimento
FROM (
    SELECT
        YEAR(order_date) as ano,
        MONTH(order_date) as mes,
        ROUND(SUM(sales)) as receita
    FROM fact_Sales
    GROUP BY ano, mes
)

ano,mes,receita,crescimento
2015,1,14206.0,null
2015,2,4520.0,-9686.0
2015,3,55206.0,50686.0
2015,4,27907.0,-27299.0
2015,5,23644.0,-4263.0
2015,6,34323.0,10679.0
2015,7,33782.0,-541.0
2015,8,27118.0,-6664.0
2015,9,81624.0,54506.0
2015,10,31453.0,-50171.0
